In [3]:
import json
import os

import joblib
import mlflow
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from dotenv import load_dotenv
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

In [7]:
# загружаем данные
data = pd.read_csv("../data/dvc_data.csv")

# выделяем таргет
y = data["price"]
X = data.drop(columns=["price"])

# делим на train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
# Выделяем категориальные признаки
cat_features = ["is_apartment", "studio", "building_type_int"]


In [8]:
# Задаем папку для логирования временных файлов при обучении
train_dir_path = "../models/logs/catboost_training"
os.makedirs(train_dir_path, exist_ok=True)
# Выбираем модель с гипперпараметрами
model = CatBoostRegressor(
        cat_features=cat_features,
        verbose=100,
        random_state=42,
        train_dir=train_dir_path,
    )

In [9]:
# Тренируем модель
model.fit(X_train, y_train)

# сохраняем обученную модель в models/train_model.pkl
os.makedirs("../models", exist_ok=True)
with open("../models/train_model.pkl", "wb") as fd:
    joblib.dump(model, fd)

    # Предсказываем
y_pred = model.predict(X_test)

# Метрики для регрессии (целевая это RMSE)
metrics_dict = {
    "rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
    "mae": mean_absolute_error(y_test, y_pred),
    "r2": r2_score(y_test, y_pred),
}

Learning rate set to 0.082574
0:	learn: 4556412.5111479	total: 119ms	remaining: 1m 59s
100:	learn: 2588837.5689403	total: 3.71s	remaining: 33s
200:	learn: 2503261.5166198	total: 8.05s	remaining: 32s
300:	learn: 2458846.8450883	total: 11.8s	remaining: 27.5s
400:	learn: 2425710.1902017	total: 15.6s	remaining: 23.3s
500:	learn: 2400048.2711932	total: 18.9s	remaining: 18.8s
600:	learn: 2376756.9093950	total: 24.9s	remaining: 16.5s
700:	learn: 2357687.0795139	total: 31.9s	remaining: 13.6s
800:	learn: 2338173.5835847	total: 36.1s	remaining: 8.96s
900:	learn: 2320206.2987559	total: 40s	remaining: 4.39s
999:	learn: 2305668.5626663	total: 44.3s	remaining: 0us


In [12]:
# сохраняем метрики в json
os.makedirs("../metrics", exist_ok=True)
with open("../metrics/fit_model_metrics.json", "w") as f:
    json.dump(metrics_dict, f, indent=2)

In [14]:
# Настройка mlflow
load_dotenv()
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net"
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY")

TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

EXPERIMENT_NAME = "yandex_estate"
RUN_NAME = "yandex_estate_run"
REGISTRY_MODEL_NAME = "yandex_estate"

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

if experiment is None:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
else:
    experiment_id = experiment.experiment_id

pip_requirements = "../../requirements.txt"
signature = mlflow.models.infer_signature(X_test, y_pred)
input_example = X_test[:10]

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id):

    mlflow.log_params(model.get_params())
    mlflow.log_metrics(metrics_dict)
    mlflow.catboost.log_model(
        cb_model=model,
        artifact_path="models",
        registered_model_name=REGISTRY_MODEL_NAME,
        signature=signature,
        input_example=input_example,
        pip_requirements=pip_requirements,
        await_registration_for=60,
    )

/home/mle-user/mle_projects/mle-project-sprint-2-v001/.venv_s2/lib/python3.10/site-packages/mlflow/models/signature.py:212: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  inputs = _infer_schema(model_input) if model_input is not None else None
Successfully registered model 'yandex_estate'.
2025/08/24 22:13:16 INFO mlflow.tracking._model_registry.client: Waiting up to 60 seco

In [ ]:

# Обучаем, логируем в MLflow и получаем метрики
metrics = train_and_log(model, X_train, X_test, y_train, y_test)

